# Database Enrichment (Imputation) — Clean, Reproducible Notebook

This notebook does **two things**:
1. **Translate coded categorical values** using `index_codes_en.csv` (optional).
2. **Impute missing values**:
   - **Continuous** targets with **XGBoost regression**
   - **Categorical** targets with a **PyTorch MLP classifier**

It also produces **reviewer-friendly diagnostics** for ML imputation:
- **SHAP** (feature attribution) for one representative continuous target
- **PDP** (partial dependence) for a couple of "engineering" drivers
- **Stratified error table** by decade (if construction year exists)

> You only need to edit the paths in the first code cell.


In [2]:

# =====================
# 0) CONFIG (EDIT ME)
# =====================
import os
import numpy as np
import pandas as pd

# Input data
PATH_DATA_RAW = r"D:/Github/Data/full_data_20241025_selected.txt"   # your raw dataset
PATH_INDEX_CODES = r"D:/Github/Data/index_codes_en.csv"            # code translation table
PATH_PREDICTORS_REG = r"D:/Github/Data/df_correlation_10_optimal_20240625.csv"  # predictors map for regression
PATH_PREDICTORS_CLS = r"D:/Github/Data/df_predictor.txt"           # predictors map for classification (3 cols per target)

# Output folder
OUT_DIR = "revision_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Whether to translate code columns using index_codes_en.csv
DO_TRANSLATE_CODES = True

# SHAP + PDP: choose ONE representative continuous target to explain
SHAP_TARGET = "living space"   # must match your column name after cleaning step below
PDP_FEATURES = ["area", "year of construction of the building yyyy"]  # pick 1-2 stable drivers (must exist in X)

RANDOM_STATE = 42


In [3]:

# =====================
# 1) LOAD + OPTIONAL TRANSLATION
# =====================
import re

df = pd.read_csv(PATH_DATA_RAW)
print("Loaded df shape:", df.shape)

if DO_TRANSLATE_CODES:
    index_code = pd.read_csv(PATH_INDEX_CODES)
    mapping = index_code.set_index('CECODID')['CODTXTLD'].to_dict()

    code_cols = [
        'canton abbreviation','building category','building class','building status',
        'energy/heat source heating 1','energy/heat source heating 2',
        'energy/heat source hot water 1','energy/heat source hot water 2',
        'heat generator heating 1','heat generator heating 2',
        'heat generator hot water 1','heat generator hot water 2'
    ]
    code_cols = [c for c in code_cols if c in df.columns]
    df[code_cols] = df[code_cols].replace(mapping)

# Clean column names: remove special characters that can break downstream tooling
df.columns = df.columns.str.replace(r'[\[\]<>]', '', regex=True)

# Quick missingness overview
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_pct.to_csv(os.path.join(OUT_DIR, "missing_percentage_before.csv"))
print("Saved missing percentage to:", os.path.join(OUT_DIR, "missing_percentage_before.csv"))
missing_pct.head(20)


C:\Users\xiong\AppData\Local\Temp\ipykernel_33220\1471790240.py:6: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PATH_DATA_RAW)


Loaded df shape: (3171854, 46)
Saved missing percentage to: revision_outputs\missing_percentage_before.csv


2hInst./Renov. hot water supply               98.270917
9aNumber of residents (EFH/ MFH)              96.994471
year of demolition of the building            96.886584
5Area-related outside air volume flow V/AE    96.882612
2iInst./Renov. heating system                 96.882328
19Floor plan type                             96.115616
18Construction building                       96.115616
building volume                               95.949025
energy reference area                         94.230787
heat generator heating 2                      85.502044
energy/heat source heating 2                  85.485776
heat generator hot water 2                    61.949226
energy/heat source hot water 2                61.945443
year of construction of the building yyyy     42.349080
number of apartments                          41.130613
floor                                         40.857366
multi-storey apartment                        40.856641
air duct length                               40

In [4]:

# =====================
# 2) LOAD PREDICTOR MAPS (REG + CLS)
# =====================
df_predictor_reg = pd.read_csv(PATH_PREDICTORS_REG)
df_predictor_reg.columns = df_predictor_reg.columns.str.replace(r'[\[\]<>]', '', regex=True)
df_predictor_reg = df_predictor_reg.replace(r'[\[\]<>]', '', regex=True)

df_predictor_cls = pd.read_csv(PATH_PREDICTORS_CLS)
df_predictor_cls.columns = df_predictor_cls.columns.str.replace(r'[\[\]<>]', '', regex=True)
df_predictor_cls = df_predictor_cls.replace(r'[\[\]<>]', '', regex=True)

print("Predictor map (reg) shape:", df_predictor_reg.shape)
print("Predictor map (cls) shape:", df_predictor_cls.shape)


Predictor map (reg) shape: (10, 22)
Predictor map (cls) shape: (3, 29)


In [5]:

# =====================
# 3) CONTINUOUS IMPUTATION (XGBoost Regression)
# =====================
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
from xgboost import plot_importance
import matplotlib.pyplot as plt

# Edit this list if needed (must match columns in df)
predictions_regression = [
    'building area','area','perimeter','height','number of floors','living space',
    'number of rooms','cooking equipment','number of radiators','air duct length',
    'water pipe length','number of toilets','number of sink','number of shower',
    'number of bathtub','electrical cable length','number of apartments',
    'year of construction of the building yyyy','2iInst./Renov. heating system',
    'year of demolition of the building','9aNumber of residents (EFH/ MFH)',
    '2hInst./Renov. hot water supply'
]
predictions_regression = [c for c in predictions_regression if c in df.columns]

plots_dir = os.path.join(OUT_DIR, "plots_regression")
os.makedirs(plots_dir, exist_ok=True)

results_rows = []

def _safe_filename(s: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', s)

for target in predictions_regression:
    # Collect predictors from mapping file (top N correlated columns stored as a column)
    if target not in df_predictor_reg.columns:
        print(f"[SKIP] target not in predictor map: {target}")
        continue

    predictors = [x for x in df_predictor_reg[target].tolist() if str(x) not in ("nan", "None")]
    predictors = [p for p in predictors if p in df.columns and p != target]

    if len(predictors) == 0:
        print(f"[SKIP] no valid predictors for: {target}")
        continue

    # Split into available vs missing for this target
    status_col = f"{target}_status"
    if status_col not in df.columns:
        df[status_col] = np.where(df[target].isna(), "missing", "available")

    df_missing = df[df[target].isna()].copy()
    df_obs = df.dropna(subset=[target]).copy()

    # IMPORTANT: drop rows with missing predictors in training
    df_obs = df_obs.dropna(subset=predictors)
    if len(df_obs) < 200:
        print(f"[SKIP] too few training rows for {target}: {len(df_obs)}")
        continue

    X = df_obs[predictors]
    y = df_obs[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    model = xgb.XGBRegressor(
        n_estimators=500, max_depth=3, learning_rate=0.1,
        random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae = float(mean_absolute_error(y_test, y_pred))

    results_rows.append({
        "target": target, "n_train": len(X_train), "n_test": len(X_test),
        "n_predictors": len(predictors), "r2": float(r2), "rmse": rmse, "mae": mae
    })

    safe = _safe_filename(target)

    # Feature importance (XGBoost built-in)
    plt.figure()
    plot_importance(model, max_num_features=15)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"feature_importance_{safe}.png"), dpi=300)
    plt.close()

    # Pred vs Actual scatter
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.5)
    min_val = float(min(y_test.min(), y_pred.min()))
    max_val = float(max(y_test.max(), y_pred.max()))
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{target} | R²={r2:.3f}, RMSE={rmse:.3f}")
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"scatter_{safe}.png"), dpi=300)
    plt.close()

    # Fill missing (only if we have missing rows and their predictors are available)
    if len(df_missing) > 0:
        df_missing_pred = df_missing.dropna(subset=predictors).copy()
        if len(df_missing_pred) > 0:
            df_missing_pred[target] = model.predict(df_missing_pred[predictors])
            df.update(df_missing_pred[[target]])

    print(f"[OK] {target}: R²={r2:.3f}, MAE={mae:.3f}, RMSE={rmse:.3f}")

# Save regression results summary
reg_results = pd.DataFrame(results_rows).sort_values("r2", ascending=False)
reg_results.to_csv(os.path.join(OUT_DIR, "regression_metrics_summary.csv"), index=False)
print("Saved:", os.path.join(OUT_DIR, "regression_metrics_summary.csv"))
reg_results.head(10)


[OK] building area: R²=0.839, MAE=42.027, RMSE=96.154
[OK] area: R²=0.948, MAE=151.877, RMSE=398.442
[OK] perimeter: R²=0.893, MAE=7.876, RMSE=30.135
[OK] height: R²=0.422, MAE=2.742, RMSE=4.321
[OK] number of floors: R²=0.764, MAE=0.499, RMSE=0.667
[OK] living space: R²=0.948, MAE=62.110, RMSE=112.084
[OK] number of rooms: R²=0.946, MAE=2.135, RMSE=4.266
[OK] cooking equipment: R²=0.992, MAE=0.051, RMSE=0.406
[OK] number of radiators: R²=0.967, MAE=0.633, RMSE=3.766
[OK] air duct length: R²=0.882, MAE=7.328, RMSE=20.677
[OK] water pipe length: R²=0.967, MAE=4.259, RMSE=35.174
[OK] number of toilets: R²=0.852, MAE=0.502, RMSE=2.046
[OK] number of sink: R²=0.852, MAE=0.502, RMSE=2.046
[OK] number of shower: R²=0.852, MAE=0.502, RMSE=2.046
[OK] number of bathtub: R²=0.852, MAE=0.502, RMSE=2.046
[OK] electrical cable length: R²=0.882, MAE=183.190, RMSE=516.925
[OK] number of apartments: R²=0.952, MAE=0.461, RMSE=1.731
[OK] year of construction of the building yyyy: R²=-0.340, MAE=29.047, 

c:\Users\xiong\anaconda3\envs\myenv\lib\site-packages\xgboost\plotting.py:113: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  _, ax = plt.subplots(1, 1)
C:\Users\xiong\AppData\Local\Temp\ipykernel_33220\1597513002.py:90: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(6, 6))


[OK] year of demolition of the building: R²=0.160, MAE=2.535, RMSE=3.465


C:\Users\xiong\AppData\Local\Temp\ipykernel_33220\1597513002.py:83: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure()


[OK] 9aNumber of residents (EFH/ MFH): R²=0.544, MAE=9.390, RMSE=20.818
[OK] 2hInst./Renov. hot water supply: R²=0.020, MAE=689.852, RMSE=827.535
Saved: revision_outputs\regression_metrics_summary.csv


...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [6]:

# =====================
# 4) ML EXPLAINABILITY FOR ONE CONTINUOUS TARGET (SHAP + PDP + STRATIFIED ERROR)
# =====================
# This section is for Reviewer #6 / Supplementary figures.
# It runs ONLY for `SHAP_TARGET` (self-contained re-train).

import shap
from sklearn.inspection import PartialDependenceDisplay

explain_dir = os.path.join(OUT_DIR, "explainability")
os.makedirs(explain_dir, exist_ok=True)

target = SHAP_TARGET
if target not in df.columns or target not in df_predictor_reg.columns:
    raise ValueError(f"SHAP_TARGET not found in df or predictor map: {target}")

predictors = [x for x in df_predictor_reg[target].tolist() if str(x) not in ("nan", "None")]
predictors = [p for p in predictors if p in df.columns and p != target]
if len(predictors) == 0:
    raise ValueError(f"No predictors found for SHAP_TARGET: {target}")

df_obs = df.dropna(subset=[target]).dropna(subset=predictors).copy()
X = df_obs[predictors]
y = df_obs[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
model = xgb.XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.1, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Explainability model R²:", round(r2_score(y_test, y_pred), 4))

# ---- SHAP (TreeExplainer) ----
X_explain = X_test.sample(n=min(2000, len(X_test)), random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_explain)

# Summary (beeswarm)
shap.summary_plot(shap_values, X_explain, show=False)
plt.tight_layout()
plt.savefig(os.path.join(explain_dir, f"shap_summary_{_safe_filename(target)}.png"), dpi=300)
plt.close()

# Bar importance
shap.summary_plot(shap_values, X_explain, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(os.path.join(explain_dir, f"shap_bar_{_safe_filename(target)}.png"), dpi=300)
plt.close()

print("Saved SHAP plots to:", explain_dir)

# ---- PDP ----
pdp_feats = [f for f in PDP_FEATURES if f in X_train.columns]
if len(pdp_feats) > 0:
    fig, ax = plt.subplots(figsize=(7, 4))
    PartialDependenceDisplay.from_estimator(model, X_train, features=pdp_feats, grid_resolution=30, ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(explain_dir, f"pdp_{_safe_filename(target)}.png"), dpi=300)
    plt.close()
    print("Saved PDP plot:", os.path.join(explain_dir, f"pdp_{_safe_filename(target)}.png"))
else:
    print("No PDP features found in X columns. Skipping PDP. Check PDP_FEATURES.")

# ---- Stratified error by construction decade (if present) ----
year_col = "year of construction of the building yyyy"
if year_col in df_obs.columns:
    decade = (np.floor(pd.to_numeric(df_obs[year_col], errors="coerce") / 10) * 10).astype("Int64").astype(str)
    decade = decade.replace("<NA>", "unknown")
    df_tmp = df_obs.loc[X_test.index, [year_col]].copy()
    df_tmp["decade"] = decade.loc[X_test.index].values
    df_tmp["y_true"] = y_test.values
    df_tmp["y_pred"] = y_pred
    df_tmp["abs_err"] = (df_tmp["y_true"] - df_tmp["y_pred"]).abs()
    tbl = df_tmp.groupby("decade").agg(n=("y_true","size"), mae=("abs_err","mean")).reset_index().sort_values("mae", ascending=False)
    tbl.to_csv(os.path.join(explain_dir, f"error_by_decade_{_safe_filename(target)}.csv"), index=False)
    print("Saved stratified error table:", os.path.join(explain_dir, f"error_by_decade_{_safe_filename(target)}.csv"))
    tbl.head(10)
else:
    print("Construction year column not found; skipping stratified error table.")


c:\Users\xiong\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Explainability model R²: 0.971
Saved SHAP plots to: revision_outputs\explainability
No PDP features found in X columns. Skipping PDP. Check PDP_FEATURES.
Saved stratified error table: revision_outputs\explainability\error_by_decade_living_space.csv


In [7]:

# =====================
# 5) CATEGORICAL IMPUTATION (PyTorch MLP Classification) — Clean Version
# =====================
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

predictions_classification = [
    'energy/heat source heating 1', 'energy/heat source heating 2',
    'energy/heat source hot water 1', 'energy/heat source hot water 2',
    'building category', 'building class', 'building status',
    'heat generator heating 1', 'heat generator heating 2',
    'heat generator hot water 1', 'heat generator hot water 2'
]
predictions_classification = [c for c in predictions_classification if c in df.columns]

class MLP(nn.Module):
    def __init__(self, n_in: int, n_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, n_out)
        )
    def forward(self, x):
        return self.net(x)

batch_size = 4096
num_epochs = 30
lr = 1e-3

cls_dir = os.path.join(OUT_DIR, "classification_models")
os.makedirs(cls_dir, exist_ok=True)

cls_metrics = []

def safe_float_df(df_in: pd.DataFrame) -> np.ndarray:
    tmp = df_in.copy()
    for c in tmp.columns:
        if tmp[c].dtype == "object":
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
    return tmp.fillna(0.0).to_numpy(dtype=np.float32)

for target in predictions_classification:
    if target not in df_predictor_cls.columns:
        print(f"[SKIP] target not in classification predictor map: {target}")
        continue

    predictors = [x for x in df_predictor_cls[target].tolist() if str(x) not in ("nan", "None")]
    predictors = [p for p in predictors if p in df.columns and p != target]
    if len(predictors) == 0:
        print(f"[SKIP] no valid predictors for: {target}")
        continue

    df_obs = df.dropna(subset=[target]).copy()
    df_miss = df[df[target].isna()].copy()

    df_obs_train = df_obs.dropna(subset=predictors).copy()
    if len(df_obs_train) < 500:
        print(f"[SKIP] too few training rows for {target}: {len(df_obs_train)}")
        continue

    X_all = safe_float_df(df_obs_train[predictors])
    le = LabelEncoder()
    y_all = le.fit_transform(df_obs_train[target].astype(str).values).astype(np.int64)

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all
    )

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    X_test_t  = torch.tensor(X_test, dtype=torch.float32)
    y_test_t  = torch.tensor(y_test, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)

    model = MLP(n_in=X_train.shape[1], n_out=len(le.classes_)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    train_acc_hist, test_acc_hist = [], []

    for epoch in range(num_epochs):
        model.train()
        correct, total = 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.numel()
        train_acc = correct / max(total, 1)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).argmax(dim=1)
                correct += (pred == yb).sum().item()
                total += yb.numel()
        test_acc = correct / max(total, 1)

        train_acc_hist.append(train_acc)
        test_acc_hist.append(test_acc)

    safe = _safe_filename(target)

    import matplotlib.pyplot as plt
    plt.figure()
    plt.plot(train_acc_hist, label="train")
    plt.plot(test_acc_hist, label="test")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title(f"Accuracy curve — {target}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(cls_dir, f"acc_curve_{safe}.png"), dpi=300)
    plt.close()

    torch.save(model.state_dict(), os.path.join(cls_dir, f"mlp_{safe}.pth"))
    pd.Series(le.classes_).to_csv(os.path.join(cls_dir, f"labels_{safe}.csv"), index=False)

    cls_metrics.append({"target": target, "n_train": len(X_train), "n_test": len(X_test), "test_acc": float(test_acc)})
    print(f"[OK] {target}: test_acc={test_acc:.3f}")

    if len(df_miss) > 0:
        df_miss_pred = df_miss.dropna(subset=predictors).copy()
        if len(df_miss_pred) > 0:
            X_miss = safe_float_df(df_miss_pred[predictors])
            X_miss_t = torch.tensor(X_miss, dtype=torch.float32).to(device)
            model.eval()
            with torch.no_grad():
                pred_idx = model(X_miss_t).argmax(dim=1).cpu().numpy()
            pred_labels = le.inverse_transform(pred_idx)
            df.loc[df_miss_pred.index, target] = pred_labels

cls_summary = pd.DataFrame(cls_metrics).sort_values("test_acc", ascending=False)
cls_summary.to_csv(os.path.join(OUT_DIR, "classification_metrics_summary.csv"), index=False)
print("Saved:", os.path.join(OUT_DIR, "classification_metrics_summary.csv"))
cls_summary.head(10)


Device: cpu


KeyboardInterrupt: 

In [ ]:

# =====================
# 6) EXPORT FINAL ENRICHED DATASET + NULL COUNTS
# =====================
out_data = os.path.join(OUT_DIR, "full_data_enriched.csv")
df.to_csv(out_data, index=False)
print("Saved enriched data:", out_data)

null_counts = df.isnull().sum().sort_values(ascending=False)
null_counts.to_csv(os.path.join(OUT_DIR, "null_values_after_enrichment.csv"))
print("Saved null counts:", os.path.join(OUT_DIR, "null_values_after_enrichment.csv"))

null_counts.head(30)
